# 🔄 זרימות עבודה בסיסיות בסוכנים עם Microsoft Foundry (Python)

## 📋 הדרכת תזמור זרימת עבודה

פנקס זה מציג את יכולות ה-**Workflow Builder** החזקות של מסגרת הסוכן של Microsoft. למד כיצד ליצור זרימות עבודה סבוכות מרובות שלבים שיכולות לטפל בתהליכים עסקיים מורכבים ולתאם מספר פעולות AI בצורה חלקה.

> **הערת הגירה:** דוגמה זו התייחסה בעבר למודלים של GitHub. מודלים של GitHub אינם נתמכים יותר (יפסיקו ביולי 2026), ולכן כעת משתמשים ב-**Microsoft Foundry** דרך `FoundryChatClient`, הפונה ל-API של Azure OpenAI **Responses**.

## 🎯 מטרות הלמידה

### 🏗️ **ארכיטקטורת זרימת עבודה**
- **Workflow Builder**: עיצוב ותזמור תהליכים מרובי שלבים מורכבים
- **ביצוע מונחה-אירועים**: טיפול באירועי זרימת עבודה ומעברי מצב
- **עיצוב זרימת עבודה חזותית**: יצירה והמחשה של מבני זרימת עבודה
- **אינטגרציה עם Microsoft Foundry**: ניצול מודלים של AI בהקשרי זרימת עבודה

### 🔄 **תזמור תהליכים**
- **פעולות רציפות**: שרשור מספר משימות סוכן בסדר לוגי
- **לוגיקה מותנית**: יישום נקודות החלטה וזרימות עבודה מסועפות
- **טיפול בשגיאות**: התאוששות שגיאות חזקה וחוסן זרימת עבודה
- **ניהול מצב**: מעקב וניהול מצב ביצוע זרימת העבודה

### 📊 **דפוסי זרימת עבודה ארגוניים**
- **אוטומציה של תהליכים עסקיים**: אוטומציה של זרימות עבודה ארגוניות מורכבות
- **תיאום מולטי-סוכנים**: תיאום בין מספר סוכנים מתמחים
- **ביצועים מדרגיים**: עיצוב זרימות עבודה לתפעול ברמת ארגון
- **ניטור ונראות**: מעקב אחר ביצועים ותוצאות זרימת העבודה

## ⚙️ דרישות מקדימות והגדרה

### 📦 **תלויות נדרשות**

התקן את מסגרת הסוכן עם יכולות זרימת עבודה:

```bash
pip install agent-framework -U
```

### 🔑 **הגדרת Microsoft Foundry**

התחבר עם Azure CLI (`az login`) כדי ש-AzureCliCredential יוכל לאמת אותך, ואז קבע את פרטי פרויקט Microsoft Foundry שלך.

**הגדרת סביבה (קובץ .env):**
```env
AZURE_AI_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-4.1-mini
```

### 🏢 **מקרי שימוש ארגוניים**

**דוגמאות לתהליכים עסקיים:**
- **קליטת לקוחות**: זרימות עבודה רב-שלביות של אימות והגדרה
- **צינור תוכן**: יצירת תוכן ממוכנת, סקירה ופרסום
- **עיבוד נתונים**: זרימות ETL עם טרנספורמציה מונעת AI
- **אבטחת איכות**: תהליכי בדיקה ואימות ממוכנים

**יתרונות זרימת העבודה:**
- 🎯 **אמינות**: ביצוע דטרמיניסטי עם התאוששות משגיאות
- 📈 **יכולת מדרגיות**: טיפול באוטומציה של תהליכים בהיקף גבוה
- 🔍 **נראות**: תשומות ביקורת ומעקב מלאים
- 🔧 **תחזוקה**: עיצוב חזותי ורכיבים מודולריים

## 🎨 דפוסי עיצוב זרימות עבודה

### מבנה בסיסי של זרימת עבודה
```mermaid
graph TD
    A[התחלה] --> B[משימה סוכן 1]
    B --> C{נקודת החלטה}
    C -->|הצלחה| D[משימה סוכן 2]
    C -->|כישלון| E[מטפל שגיאות]
    D --> F[סיום]
    E --> F
```

**רכיבים מרכזיים:**
- **WorkflowBuilder**: מנוע התזמור הראשי
- **WorkflowEvent**: טיפול באירועים ותקשורת
- **WorkflowViz**: המחשה חזותית ו-debug של זרימת העבודה

בואו נבנה את זרימת העבודה החכמה הראשונה שלכם! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
# Core components for building sophisticated agent workflows
from agent_framework import WorkflowBuilder, WorkflowEvent, WorkflowViz
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential


In [ ]:
# 📦 Import Environment and System Utilities
# Essential libraries for configuration and environment management

import os                      # 🔧 Environment variable access
from dotenv import load_dotenv # 📁 Secure configuration loading

In [ ]:
# 🔧 Initialize Environment Configuration
# Load Microsoft Foundry project settings from .env file
load_dotenv()


In [ ]:
# Configure the Microsoft Foundry client with keyless authentication.
# FoundryChatClient targets the Azure OpenAI Responses API.
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


In [ ]:
REVIEWER_NAME = "Concierge"
REVIEWER_INSTRUCTIONS = """
    You are an are hotel concierge who has opinions about providing the most local and authentic experiences for travelers.
    The goal is to determine if the front desk travel agent has recommended the best non-touristy experience for a traveler.
    If so, state that it is approved.
    If not, provide insight on how to refine the recommendation without using a specific example. 
    """

In [ ]:
FRONTDESK_NAME = "FrontDesk"
FRONTDESK_INSTRUCTIONS = """
    You are a Front Desk Travel Agent with ten years of experience and are known for brevity as you deal with many customers.
    The goal is to provide the best activities and locations for a traveler to visit.
    Only provide a single recommendation per response.
    You're laser focused on the goal at hand.
    Don't waste time with chit chat.
    Consider suggestions when refining an idea.
    """

In [ ]:
reviewer_agent = provider.as_agent(
    name=REVIEWER_NAME,
    instructions=REVIEWER_INSTRUCTIONS,
)

front_desk_agent = provider.as_agent(
    name=FRONTDESK_NAME,
    instructions=FRONTDESK_INSTRUCTIONS,
)


In [ ]:
workflow = (
    WorkflowBuilder(start_executor=front_desk_agent)
    .add_edge(front_desk_agent, reviewer_agent)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
# Display the exported workflow SVG inline in the notebook

from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")


In [ ]:
# Workflow.run_stream is no longer part of the public API; the current Workflow
# returns a results object whose `get_outputs()` produces the AgentResponse from
# each output executor. The reviewer (last stage) is the only output here.
events = await workflow.run("I would like to go to Paris.")
outputs = events.get_outputs()
result = outputs[0].text if outputs else ""

In [ ]:
result.replace("None", "")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
